# 08 — Mutable and Immutable Objects

## Learning objectives

By the end of this notebook, you should be able to:

- Explain the difference between mutable and immutable objects.
- Distinguish a variable name from the object it refers to.
- Predict what happens when two variables refer to the same object.
- Understand mutation versus reassignment.
- Recognize common mutable and immutable built-in types.
- Avoid accidental changes caused by shared references.
- Use shallow copies and understand nested mutable objects.

**How to use this notebook:** Run the code cells from top to bottom. Each section builds on the previous one. Try the exercises before reading the solutions.

## 1. Variables refer to objects

A variable is a name that refers to an object. The name is not the object itself.

In the example below, `first` refers to an integer object. When we write `second = first`, Python makes `second` refer to the same integer object. Integers are immutable, so an operation that appears to “change” an integer actually creates or refers to another integer and reassigns the name.

In [1]:
first = 10
second = first

print("first:", first)
print("second:", second)
print("Same object?", first is second)

first: 10
second: 10
Same object? True


## 2. What does mutable mean?

A **mutable** object can be changed after it is created. A **immutable** object cannot be changed after it is created.

Common mutable built-in types include:

- `list`
- `dict`
- `set`
- Most user-defined class instances

Common immutable built-in types include:

- `int`, `float`, `complex`
- `bool`
- `str`
- `tuple` (with an important caveat for mutable elements inside it)
- `frozenset`
- `bytes`

The following examples show the basic difference.

In [2]:
# A list is mutable: we can change one of its elements.
numbers = [10, 20, 30]
numbers[0] = 99
print(numbers)

# A string is immutable: its characters cannot be assigned individually.
word = "cat"
try:
    word[0] = "b"
except TypeError as error:
    print(type(error).__name__ + ":", error)

[99, 20, 30]
TypeError: 'str' object does not support item assignment


## 3. Mutation versus reassignment

These are two different operations:

- **Mutation:** change the existing object.
- **Reassignment:** make a variable name refer to another object.

Changing a list element mutates the existing list. Assigning a new list to a variable reassigns that name.

In [3]:
values = [1, 2, 3]
alias = values

values.append(4)       # Mutation: changes the shared list.
print("After mutation:")
print("values:", values)
print("alias:", alias)
print("Same list?", values is alias)

values = [7, 8]        # Reassignment: values now refers to a different list.
print("\nAfter reassignment:")
print("values:", values)
print("alias:", alias)
print("Same list?", values is alias)

After mutation:
values: [1, 2, 3, 4]
alias: [1, 2, 3, 4]
Same list? True

After reassignment:
values: [7, 8]
alias: [1, 2, 3, 4]
Same list? False


## 4. Two names, one mutable object

If two variables refer to the same mutable object, a mutation through either variable is visible through both names.

In [4]:
shopping = ["bread", "milk"]
another_name = shopping

another_name.append("eggs")

print("shopping:", shopping)
print("another_name:", another_name)
print("Same object?", shopping is another_name)

shopping: ['bread', 'milk', 'eggs']
another_name: ['bread', 'milk', 'eggs']
Same object? True


## 5. Two separate objects with the same contents

Two lists can contain equal values while still being different objects. `==` compares contents for lists; `is` checks identity.

In [5]:
a = [1, 2, 3]
b = [1, 2, 3]

print("Equal contents?", a == b)
print("Same object?", a is b)

b.append(4)
print("a after changing b:", a)
print("b after changing b:", b)

Equal contents? True
Same object? False
a after changing b: [1, 2, 3]
b after changing b: [1, 2, 3, 4]


## 6. Reassignment with immutable objects

Integers and strings are immutable. An expression such as `count = count + 1` does not modify the original integer object; it calculates a result and reassigns the name `count`.

In [6]:
count = 5
alias = count

count = count + 1

print("count:", count)
print("alias:", alias)
print("Same object now?", count is alias)

count: 6
alias: 5
Same object now? False


## 7. Why a function can change a list

When you pass a list to a function, the parameter refers to the same list object. Mutating that list inside the function is visible to the caller.

In [7]:
def add_item(items, item):
    items.append(item)

tasks = ["study Python"]
add_item(tasks, "practice lists")

print(tasks)

['study Python', 'practice lists']


## 8. Rebinding a function parameter does not reassign the caller's variable

A function parameter is a local name. If you assign a new list to that parameter, the caller's variable continues to refer to its original list. This is different from mutating the original list.

In [8]:
def replace_list(items):
    items = ["new", "list"]
    print("Inside function:", items)

original = ["original", "items"]
replace_list(original)
print("Outside function:", original)

Inside function: ['new', 'list']
Outside function: ['original', 'items']


## 9. Mutable default argument pitfall

Default argument expressions are evaluated once when the function is defined, not every time it is called. A mutable default list can therefore retain items between calls.

This example intentionally demonstrates the problem. The safer pattern is shown next.

In [9]:
def add_tag(tag, tags=[]):
    tags.append(tag)
    return tags

print(add_tag("python"))
print(add_tag("ml"))
print(add_tag("devops"))

['python']
['python', 'ml']
['python', 'ml', 'devops']


### Safer pattern: use `None` as the default

Use `None` to indicate that the caller did not provide a list, then create a fresh list inside the function.

In [10]:
def add_tag_safely(tag, tags=None):
    if tags is None:
        tags = []
    tags.append(tag)
    return tags

print(add_tag_safely("python"))
print(add_tag_safely("ml"))
print(add_tag_safely("devops"))

existing_tags = ["backend"]
print(add_tag_safely("testing", existing_tags))
print("Original list:", existing_tags)

['python']
['ml']
['devops']
['backend', 'testing']
Original list: ['backend', 'testing']


## 10. Copying a list

A shallow copy creates a new outer list. For a flat list, changes to the copied list's elements do not change the original list. Common shallow-copy methods include `list.copy()`, `list(original)`, and `original[:]`.

In [11]:
original = [10, 20, 30]
copied = original.copy()

print("Equal?", original == copied)
print("Same object?", original is copied)

copied.append(40)
print("original:", original)
print("copied:", copied)

Equal? True
Same object? False
original: [10, 20, 30]
copied: [10, 20, 30, 40]


## 11. Shallow copies and nested objects

A shallow copy duplicates only the outer container. If the list contains another mutable object, both lists may still refer to the same inner object.

In [12]:
original = [["a", "b"], ["c", "d"]]
copied = original.copy()

print("Outer list is same?", original is copied)
print("First inner list is same?", original[0] is copied[0])

copied[0].append("changed")
print("original:", original)
print("copied:", copied)

Outer list is same? False
First inner list is same? True
original: [['a', 'b', 'changed'], ['c', 'd']]
copied: [['a', 'b', 'changed'], ['c', 'd']]


### Deep copy for nested mutable objects

`copy.deepcopy()` recursively copies nested objects in typical container structures. Use it when you need an independent nested structure, while keeping in mind that some custom objects need special handling.

In [13]:
import copy

original = [["a", "b"], ["c", "d"]]
deep_copied = copy.deepcopy(original)

deep_copied[0].append("new")

print("original:", original)
print("deep_copied:", deep_copied)
print("First inner list is same?", original[0] is deep_copied[0])

original: [['a', 'b'], ['c', 'd']]
deep_copied: [['a', 'b', 'new'], ['c', 'd']]
First inner list is same? False


## 12. Tuples are immutable, but their contents may be mutable

You cannot replace a tuple's element, but if an element is a mutable object such as a list, that inner list can still change.

In [14]:
point = (10, 20)
try:
    point[0] = 99
except TypeError as error:
    print(type(error).__name__ + ":", error)

mixed = ("labels", ["python", "ml"])
mixed[1].append("devops")
print(mixed)

TypeError: 'tuple' object does not support item assignment
('labels', ['python', 'ml', 'devops'])


## 13. Immutable values can be useful as dictionary keys

Dictionary keys must be hashable. Common immutable types such as strings, integers, and tuples containing only hashable values can be keys. Lists and dictionaries are mutable and cannot be used as dictionary keys.

In [15]:
scores = {
    "Asha": 90,
    101: 85,
    (2, 3): 95,
}
print(scores["Asha"])
print(scores[101])
print(scores[(2, 3)])

try:
    invalid = {[1, 2]: "value"}
except TypeError as error:
    print(type(error).__name__ + ":", error)

90
85
95
TypeError: cannot use 'list' as a dict key (unhashable type: 'list')


## 14. Common mistakes

1. **Assuming `b = a` copies a list.** It creates another reference to the same list.
2. **Confusing `==` with `is`.** Equality compares values; identity compares whether they are the same object.
3. **Thinking reassignment mutates the previous object.** Reassignment changes what a name refers to.
4. **Using a mutable default argument unintentionally.** Use `None` and create a fresh object inside the function.
5. **Assuming a shallow copy copies nested lists too.** It copies only the outer container.
6. **Thinking a tuple makes every object inside it immutable.** A tuple cannot replace its elements, but a nested list can still be mutated.

In [16]:
a = [[1, 2]]
b = a
c = a.copy()

print("a == b:", a == b)
print("a is b:", a is b)
print("a == c:", a == c)
print("a is c:", a is c)

c[0].append(3)
print("a after inner mutation through c:", a)

a == b: True
a is b: True
a == c: True
a is c: False
a after inner mutation through c: [[1, 2, 3]]


## 15. Practice exercises

Try these before reading the solutions.

1. Create a list `a = [1, 2]`, assign `b = a`, append `3` through `b`, and predict `a`.
2. Create two separate lists with equal contents. Show the difference between `==` and `is`.
3. Write a function that appends an item to a list. Demonstrate that the caller's list changes.
4. Write a function that returns a new list when its optional list argument is `None`.
5. Make a shallow copy of a nested list and demonstrate that mutating an inner list affects both outer lists.
6. Use `copy.deepcopy()` to make the nested lists independent.

In [17]:
# Exercise 1
a = [1, 2]
b = a
b.append(3)

print("a:", a)
print("b:", b)

a: [1, 2, 3]
b: [1, 2, 3]


In [18]:
# Exercise 2
first = [4, 5]
second = [4, 5]

print(first == second)
print(first is second)

True
False


In [19]:
# Exercise 3
def append_value(values, value):
    values.append(value)

data = [10]
append_value(data, 20)
print(data)

[10, 20]


In [20]:
# Exercise 4
def make_list(values=None):
    if values is None:
        values = []
    return values

one = make_list()
two = make_list()

one.append("only in one")
print("one:", one)
print("two:", two)
print("Different objects?", one is not two)

one: ['only in one']
two: []
Different objects? True


In [21]:
# Exercise 5
nested = [[1], [2]]
shallow = nested.copy()

shallow[0].append(99)
print("nested:", nested)
print("shallow:", shallow)

nested: [[1, 99], [2]]
shallow: [[1, 99], [2]]


In [22]:
# Exercise 6
import copy

nested = [[1], [2]]
independent = copy.deepcopy(nested)

independent[0].append(99)
print("nested:", nested)
print("independent:", independent)

nested: [[1], [2]]
independent: [[1, 99], [2]]


## 16. Review questions

1. What is the difference between mutation and reassignment?
2. What happens when `b = a` is used with a list?
3. How are `==` and `is` different?
4. Why can a mutable default argument cause unexpected behavior?
5. What does a shallow copy duplicate, and what can it still share?
6. Why can a list inside a tuple still be changed?
7. Why are lists generally unsuitable as dictionary keys?

**Key takeaway:** Variables are names that refer to objects. When you understand whether an object is mutable and whether two names refer to the same object, many surprising Python behaviors become predictable.